# LeetCode #63 — Unique Paths II: DP trên lưới có vật cản

**Mức độ:** Medium · **Chủ đề:** Array, Dynamic Programming, Matrix

Skeleton trên LeetCode:

```python
class Solution:
    def uniquePathsWithObstacles(self, obstacleGrid: List[List[int]]) -> int:
```

Bài #62 đếm đường đi trên lưới trống. Bài này thêm **vật cản** — công thức tổ hợp không dùng được nữa, nhưng DP vẫn chạy với **một thay đổi nhỏ**: ô có vật cản có **0** đường đi.

Mục tiêu của notebook:

1. Điều chỉnh công thức DP cho ô bị chặn.
2. Xử lý **hàng đầu / cột đầu** khi có vật cản: sau vật cản, mọi ô đều **không tới được**.
3. Cách khởi tạo gọn: `dp[0][0] = 1` và duyệt mọi ô với cùng một công thức.
4. Tối ưu bộ nhớ một hàng.

# 1. Bài toán là gì?

Robot ở góc trên trái của lưới `m × n`, chỉ đi **phải** hoặc **xuống**, muốn tới góc dưới phải. `obstacleGrid[i][j] == 1` là **vật cản** (không được đi vào), `0` là ô trống.

Đếm số đường đi. Đáp án không vượt `2 * 10^9`.

**Constraints (theo đề):**

- `1 <= m, n <= 100`
- `obstacleGrid[i][j]` là `0` hoặc `1`.

---

## Ví dụ

```text
0 0 0
0 1 0      ->  2     (vòng trên-phải hoặc vòng trái-dưới)
0 0 0

0 1
0 0        ->  1
```

Nếu ô **xuất phát** hoặc ô **đích** là vật cản → đáp án `0`.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quy hoạch động (Dynamic Programming — DP)
2. DP hai chiều (bảng `dp[i][j]`)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [1]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [2]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.2 DP hai chiều (bảng `dp[i][j]`)

### Khi nào cần bảng hai chiều?

Khi trạng thái cần **hai** thông số, thường gặp ở:

- **Lưới**: `dp[i][j]` = đáp án cho ô `(i, j)` (số đường đi, tổng nhỏ nhất…).
- **Hai chuỗi**: `dp[i][j]` = đáp án cho **tiền tố** `s[:i]` và `t[:j]` (edit distance, LCS, xen kẽ…).
- **Đoạn con**: `dp[i][j]` = đáp án cho đoạn `s[i..j]` (palindrome…).

### Ví dụ 1: số đường đi trên lưới

Robot ở góc trên-trái, mỗi bước đi **phải** hoặc **xuống**. Có bao nhiêu đường tới ô `(i, j)`? Bước cuối đến từ ô **trên** hoặc ô **trái**:

```text
dp[i][j] = dp[i-1][j] + dp[i][j-1]
hàng 0 và cột 0 đều = 1 (chỉ một cách: đi thẳng)
```

In [3]:
def so_duong(m, n):
    dp = [[0] * n for _ in range(m)]      # KHÔNG dùng [[0] * n] * m
    for i in range(m):
        for j in range(n):
            if i == 0 or j == 0:
                dp[i][j] = 1
            else:
                dp[i][j] = dp[i - 1][j] + dp[i][j - 1]
    return dp

for row in so_duong(3, 5):
    print(" ".join(f"{v:>3}" for v in row))

  1   1   1   1   1
  1   2   3   4   5
  1   3   6  10  15


Thứ tự điền: theo hàng từ trên xuống, trong hàng từ trái sang → khi tính `dp[i][j]`, ô trên và ô trái **đã có**.

**Tiết kiệm bộ nhớ:** hàng `i` chỉ cần hàng `i−1` → giữ một mảng một chiều `row`, cập nhật `row[j] += row[j-1]` (giá trị cũ của `row[j]` chính là ô trên).

In [4]:
def so_duong_mot_hang(m, n):
    row = [1] * n
    for _ in range(1, m):
        for j in range(1, n):
            row[j] += row[j - 1]          # row[j] (cũ) = trên, row[j-1] (mới) = trái
    return row[-1]
print("so_duong_mot_hang(3, 7) =", so_duong_mot_hang(3, 7), "| so_duong(3, 7) =", so_duong(3, 7)[-1][-1])

so_duong_mot_hang(3, 7) = 28 | so_duong(3, 7) = 28


### Ví dụ 2: hai chuỗi — dãy con chung dài nhất (LCS)

`dp[i][j]` = độ dài dãy con chung dài nhất của `s[:i]` và `t[:j]`. Xét ký tự **cuối** của hai tiền tố:

- `s[i-1] == t[j-1]` → dùng nó: `dp[i][j] = dp[i-1][j-1] + 1`.
- khác → bỏ ký tự cuối của một trong hai: `dp[i][j] = max(dp[i-1][j], dp[i][j-1])`.

Hàng 0 và cột 0 ứng với **chuỗi rỗng** → bằng 0. Đó là lý do bảng có kích thước `(n+1) × (m+1)`.

In [5]:
def lcs(s, t):
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp

s_demo, t_demo = "abcde", "ace"
bang = lcs(s_demo, t_demo)
print("      ''  " + "   ".join(t_demo))
for i, row in enumerate(bang):
    print(f"{('' if i == 0 else s_demo[i - 1])!r:>4} " + " ".join(f"{v:>3}" for v in row))
print("LCS =", bang[-1][-1])

      ''  a   c   e
  ''   0   0   0   0
 'a'   0   1   1   1
 'b'   0   1   1   1
 'c'   0   1   2   2
 'd'   0   1   2   2
 'e'   0   1   2   3
LCS = 3


### Ví dụ 3: khoảng cách sửa (edit distance) — số phép chèn/xóa/thay ít nhất

`dp[i][j]` = số phép sửa ít nhất biến `s[:i]` thành `t[:j]`.

- Ký tự cuối bằng nhau → `dp[i-1][j-1]`.
- Khác → `1 + min(thay: dp[i-1][j-1], xóa s[i-1]: dp[i-1][j], chèn t[j-1]: dp[i][j-1])`.
- Cơ sở: `dp[i][0] = i` (xóa hết), `dp[0][j] = j` (chèn hết).

In [6]:
def khoang_cach_sua(s, t):
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]

for a_, b_ in [("horse", "ros"), ("kitten", "sitting"), ("", "abc")]:
    print(f"{a_!r} → {b_!r}: {khoang_cach_sua(a_, b_)}")

'horse' → 'ros': 3
'kitten' → 'sitting': 3
'' → 'abc': 3


### Lỗi hay gặp

- `[[0] * n] * m` tạo `m` hàng là **cùng một** list → sửa một ô thì cả cột đổi.
- Lệch chỉ số: bảng `(n+1)×(m+1)` nhưng ký tự là `s[i-1]`, `t[j-1]`.
- Quên hàng/cột cơ sở (chuỗi rỗng, biên lưới).
- Khi tối ưu một hàng: ghi đè giá trị cần dùng lại (ví dụ `dp[i-1][j-1]`) — phải lưu tạm trước.

In [7]:
sai = [[0] * 3] * 2
sai[0][0] = 9
dung = [[0] * 3 for _ in range(2)]
dung[0][0] = 9
print("[[0]*3]*2 sau khi gán ô (0,0):", sai, "| tạo bằng vòng for:", dung)

[[0]*3]*2 sau khi gán ô (0,0): [[9, 0, 0], [9, 0, 0]] | tạo bằng vòng for: [[9, 0, 0], [0, 0, 0]]


## 2.3 Áp dụng vào bài này

**DP trên lưới** như #62, thêm điều kiện: ô có vật cản thì `dp = 0` (không có đường đi qua nó). Chú ý hàng/cột đầu: sau vật cản đầu tiên, các ô phía sau trên cùng hàng/cột đều bằng 0.

# 3. Công thức DP có vật cản

Gọi `dp[r][c]` = số đường tới ô `(r, c)`.

```text
nếu grid[r][c] == 1:  dp[r][c] = 0
ngược lại:            dp[r][c] = (dp[r-1][c] nếu r > 0) + (dp[r][c-1] nếu c > 0)
dp[0][0] = 1 nếu ô đầu trống
```

Vật cản "cắt" mọi đường đi qua nó: nó đóng góp **0** cho các ô bên phải và bên dưới.

## Cái bẫy của hàng đầu / cột đầu

Ở bài #62, hàng 0 toàn số `1`. Ở đây, hàng 0 là `1` **cho tới vật cản đầu tiên**, sau đó là `0` (không thể vòng qua vì chỉ đi phải/xuống).

```text
grid hàng 0:  0 0 1 0 0
dp hàng 0:    1 1 0 0 0
```

Nếu duyệt mọi ô bằng **cùng một công thức** (không đặt sẵn hàng 0 = 1), cái bẫy này tự được xử lý.

In [8]:
from typing import List

def unique_paths_obstacles_table(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    dp = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 1:
                dp[r][c] = 0
            elif r == 0 and c == 0:
                dp[r][c] = 1
            else:
                top = dp[r - 1][c] if r > 0 else 0
                left = dp[r][c - 1] if c > 0 else 0
                dp[r][c] = top + left
    return dp[m - 1][n - 1]

print(unique_paths_obstacles_table([[0, 0, 0], [0, 1, 0], [0, 0, 0]]))
print(unique_paths_obstacles_table([[0, 1], [0, 0]]))
print(unique_paths_obstacles_table([[0, 0, 1, 0, 0]]))

2
1
0


# 4. Mô phỏng: in bảng DP

In [9]:
def show_dp(grid: List[List[int]]):
    m, n = len(grid), len(grid[0])
    dp = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 1:
                dp[r][c] = 0
            elif r == 0 and c == 0:
                dp[r][c] = 1
            else:
                dp[r][c] = (dp[r - 1][c] if r > 0 else 0) + (dp[r][c - 1] if c > 0 else 0)
    print("grid (X = vật cản)        dp")
    w = max(len(str(x)) for row in dp for x in row)
    for r in range(m):
        g = " ".join("X" if v else "." for v in grid[r])
        d = " ".join(("X" if grid[r][c] else str(dp[r][c])).rjust(w) for c in range(n))
        print(f"{g:<25} {d}")
    print("Đáp án:", dp[-1][-1])

show_dp([[0, 0, 0], [0, 1, 0], [0, 0, 0]])
print()
show_dp([[0, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [1, 0, 0, 0]])

grid (X = vật cản)        dp
. . .                     1 1 1
. X .                     1 X 1
. . .                     1 1 2
Đáp án: 2

grid (X = vật cản)        dp
. . . .                   1 1 1 1
. X . .                   1 X 1 2
. . . X                   1 1 2 X
X . . .                   X 1 3 3
Đáp án: 3


# 5. Đối chiếu: đếm bằng DFS (cho lưới nhỏ)

Để kiểm tra, viết một hàm đệ quy đơn giản (chậm nhưng chắc chắn đúng): từ `(r, c)`, nếu là vật cản → 0; nếu là đích → 1; ngược lại cộng hai hướng.

In [10]:
def count_dfs(grid: List[List[int]], r: int = 0, c: int = 0) -> int:
    m, n = len(grid), len(grid[0])
    if r >= m or c >= n or grid[r][c] == 1:
        return 0
    if r == m - 1 and c == n - 1:
        return 1
    return count_dfs(grid, r + 1, c) + count_dfs(grid, r, c + 1)

print(count_dfs([[0, 0, 0], [0, 1, 0], [0, 0, 0]]))

2


# 6. Tối ưu bộ nhớ: một hàng

Như bài #62, `row[c]` trước khi cập nhật là giá trị **hàng trên**, `row[c - 1]` là giá trị **hàng hiện tại**:

```python
row = [0] * n
row[0] = 1                         # "người" đứng trước ô (0, 0)
for each hàng r:
    for c in range(n):
        if grid[r][c] == 1: row[c] = 0
        elif c > 0:         row[c] += row[c - 1]
```

Với `c = 0`, `row[0]` giữ nguyên (chỉ đi xuống từ trên) — trừ khi gặp vật cản thì thành `0` **vĩnh viễn** cho các hàng dưới.

Mẹo `row[0] = 1` hoạt động cả khi `grid[0][0] == 1`: ngay ô đầu, nó bị đặt về `0`.

In [11]:
def unique_paths_obstacles(grid: List[List[int]]) -> int:
    n = len(grid[0])
    row = [0] * n
    row[0] = 1
    for r in range(len(grid)):
        for c in range(n):
            if grid[r][c] == 1:
                row[c] = 0
            elif c > 0:
                row[c] += row[c - 1]
    return row[-1]

print(unique_paths_obstacles([[0, 0, 0], [0, 1, 0], [0, 0, 0]]))
print(unique_paths_obstacles([[0, 1], [0, 0]]))
print(unique_paths_obstacles([[1]]))

2
1
0


# 7. Phiên bản đúng format LeetCode

In [12]:
class Solution:
    def uniquePathsWithObstacles(self, obstacleGrid: List[List[int]]) -> int:
        n = len(obstacleGrid[0])
        row = [0] * n
        row[0] = 1
        for grid_row in obstacleGrid:
            for c in range(n):
                if grid_row[c] == 1:
                    row[c] = 0
                elif c > 0:
                    row[c] += row[c - 1]
        return row[-1]

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.uniquePathsWithObstacles([[0, 0, 0], [0, 1, 0], [0, 0, 0]]), 2)
check("Ví dụ 2", sol.uniquePathsWithObstacles([[0, 1], [0, 0]]), 1)
check("Ô đầu bị chặn", sol.uniquePathsWithObstacles([[1, 0], [0, 0]]), 0)
check("Ô đích bị chặn", sol.uniquePathsWithObstacles([[0, 0], [0, 1]]), 0)
check("1 x 1 trống", sol.uniquePathsWithObstacles([[0]]), 1)
check("1 x 1 vật cản", sol.uniquePathsWithObstacles([[1]]), 0)
check("Một hàng bị chặn giữa", sol.uniquePathsWithObstacles([[0, 1, 0]]), 0)
check("Một cột", sol.uniquePathsWithObstacles([[0], [0], [0]]), 1)
check("Không vật cản = bài #62", sol.uniquePathsWithObstacles([[0] * 7 for _ in range(3)]), 28)

import random
random.seed(63)
for _ in range(3000):
    m, n = random.randint(1, 6), random.randint(1, 6)
    grid = [[1 if random.random() < 0.25 else 0 for _ in range(n)] for _ in range(m)]
    expected = count_dfs(grid)
    assert sol.uniquePathsWithObstacles(grid) == expected, grid
    assert unique_paths_obstacles_table(grid) == expected, grid
print("3000 lưới ngẫu nhiên: hai bản DP khớp DFS")

Ví dụ 1                      PASS  2
Ví dụ 2                      PASS  1
Ô đầu bị chặn                PASS  0
Ô đích bị chặn               PASS  0
1 x 1 trống                  PASS  1
1 x 1 vật cản                PASS  0
Một hàng bị chặn giữa        PASS  0
Một cột                      PASS  1
Không vật cản = bài #62      PASS  28


3000 lưới ngẫu nhiên: hai bản DP khớp DFS


# 8. Vì sao đúng? Độ phức tạp?

**Đúng:** đường đi hợp lệ tới ô trống `(r, c)` có bước cuối từ trên hoặc từ trái (hai tập rời nhau); đường đi tới ô vật cản không tồn tại (0). Quy nạp theo thứ tự duyệt cho ra mọi ô. Ô `(0, 0)` có 1 đường nếu trống, 0 nếu bị chặn.

```text
Time:  O(m · n)
Space: O(n)  (bản một hàng)  |  O(m · n) (bản bảng)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Khởi tạo cả hàng 0 / cột 0 bằng 1 như bài #62

In [14]:
def wrong_init_ones(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    dp = [[1] * n for _ in range(m)]                   # SAI: bỏ qua vật cản ở hàng/cột đầu
    for r in range(1, m):
        for c in range(1, n):
            dp[r][c] = 0 if grid[r][c] else dp[r - 1][c] + dp[r][c - 1]
    return dp[-1][-1]

g = [[0, 1, 0], [0, 0, 0]]
print("Sai :", wrong_init_ones(g), "| Đúng:", sol.uniquePathsWithObstacles(g))

Sai :

 3 | Đúng: 1


## Lỗi 2: Chỉ đặt 0 cho ô vật cản ở hàng 0, quên các ô **sau** nó

Ô sau vật cản ở hàng 0 cũng **không tới được** (chỉ có thể đến từ bên trái, mà bên trái bị chặn).

## Lỗi 3: Quên trường hợp ô đầu / ô đích là vật cản

Với cách "đặt `row[0] = 1` rồi duyệt", cả hai tự động ra 0. Nhưng nếu bạn viết `dp[0][0] = 1` **sau** khi xét vật cản, sẽ sai.

## Lỗi 4: Dùng công thức tổ hợp

`C(m+n-2, m-1)` chỉ đúng khi **không** có vật cản.

# 10. Mẹo nhận diện pattern

- DP trên lưới + **ô cấm** → ô cấm có giá trị trung hòa (0 với bài đếm, `∞` với bài tìm min).
- Duyệt mọi ô bằng **một công thức thống nhất** (kiểm tra `r > 0`, `c > 0`) giúp tránh lỗi ở hàng/cột đầu.
- Kiểm tra DP bằng **brute force DFS** trên lưới nhỏ.

Bài liên quan: **#62 Unique Paths**, **#64 Minimum Path Sum**, **#980 Unique Paths III** (Hard, đi qua mọi ô — backtracking), **#1301 Number of Paths with Max Score**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng DP:

```text
0 0 0 0
1 0 1 0
0 0 0 0
```

## Bài 2 — Điền chỗ trống (bản một hàng)

In [15]:
def exercise_2_unique_paths_obstacles(grid: List[List[int]]) -> int:
    n = len(grid[0])
    row = [0] * n
    row[0] = 1
    for r in range(len(grid)):
        for c in range(n):
            # TODO: vật cản -> 0; ngược lại nếu c > 0 thì cộng row[c - 1]
            pass
    return row[-1]


# assert exercise_2_unique_paths_obstacles([[0, 0, 0], [0, 1, 0], [0, 0, 0]]) == 2
# assert exercise_2_unique_paths_obstacles([[1]]) == 0
# assert exercise_2_unique_paths_obstacles([[0, 1, 0]]) == 0

## Bài 3 — Biến thể: lấy đường đi modulo

Lưới lớn tới `1000 × 1000` có thể có số đường cực lớn. Trả về số đường `mod 10^9 + 7`.

```python
paths_mod([[0] * 1000 for _ in range(1000)]) == math.comb(1998, 999) % (10**9 + 7)
```

In [16]:
def exercise_3_paths_mod(grid: List[List[int]]) -> int:
    # TODO
    pass


# from math import comb
# assert exercise_3_paths_mod([[0, 0, 0], [0, 1, 0], [0, 0, 0]]) == 2
# assert exercise_3_paths_mod([[0] * 300 for _ in range(300)]) == comb(598, 299) % (10**9 + 7)

## Bài 4 — Biến thể: có tới được đích không?

Trả về `True` nếu có ít nhất một đường. Bản boolean của DP.

In [17]:
def exercise_4_can_reach(grid: List[List[int]]) -> bool:
    # TODO
    pass


# assert exercise_4_can_reach([[0, 0, 0], [0, 1, 0], [0, 0, 0]]) is True
# assert exercise_4_can_reach([[0, 1], [1, 0]]) is False
# assert exercise_4_can_reach([[0]]) is True

## Bài 5 — Mini challenge: chặn ít vật cản nhất

Một lưới trống `m × n` (`m, n >= 2`). Cần đặt **ít nhất bao nhiêu** vật cản (không đặt ở ô đầu và ô đích) để robot không tới được đích?

Viết hàm brute force kiểm tra với lưới nhỏ, và đoán quy luật.

```python
min_blocks(2, 2) -> 2
min_blocks(3, 3) -> 2
```

In [18]:
def exercise_5_min_blocks(m: int, n: int) -> int:
    # TODO
    pass


# assert exercise_5_min_blocks(2, 2) == 2
# assert exercise_5_min_blocks(3, 3) == 2
# assert exercise_5_min_blocks(2, 4) == 2

# 12. Gợi ý

**Bài 1**
- Mức 1: cột 0: `1, 0, 0` (bị chặn ở hàng 1 nên mọi ô dưới bằng 0).

**Bài 3**
- Mức 1: như bản một hàng, thêm `% MOD` sau mỗi phép cộng.

**Bài 4**
- Mức 1: thay `+` bằng `or`, `0/1` bằng `False/True`.

**Bài 5**
- Mức 1: thử mọi tập con các ô (trừ đầu và đích) theo kích thước tăng dần, dùng Bài 4 để kiểm tra.
- Mức 2: luôn đủ khi chặn **hai ô kề ô đầu** (`(0,1)` và `(1,0)`) → đáp án không quá 2.
- Mức 3: chặn **một** ô có đủ không? Chỉ khi mọi đường đều đi qua ô đó.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [19]:
show_dp([[0, 0, 0, 0], [1, 0, 1, 0], [0, 0, 0, 0]])

grid (X = vật cản)        dp
. . . .                   1 1 1 1
X . X .                   X 1 X 1
. . . .                   0 1 1 2
Đáp án: 2


## Đáp án Bài 2

In [20]:
def answer_2_unique_paths_obstacles(grid: List[List[int]]) -> int:
    n = len(grid[0])
    row = [0] * n
    row[0] = 1
    for r in range(len(grid)):
        for c in range(n):
            if grid[r][c] == 1:
                row[c] = 0
            elif c > 0:
                row[c] += row[c - 1]
    return row[-1]

assert answer_2_unique_paths_obstacles([[0, 0, 0], [0, 1, 0], [0, 0, 0]]) == 2
assert answer_2_unique_paths_obstacles([[1]]) == 0
assert answer_2_unique_paths_obstacles([[0, 1, 0]]) == 0
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [21]:
from math import comb

def answer_3_paths_mod(grid: List[List[int]]) -> int:
    MOD = 10**9 + 7
    n = len(grid[0])
    row = [0] * n
    row[0] = 1
    for grid_row in grid:
        for c in range(n):
            if grid_row[c] == 1:
                row[c] = 0
            elif c > 0:
                row[c] = (row[c] + row[c - 1]) % MOD
    return row[-1]

assert answer_3_paths_mod([[0, 0, 0], [0, 1, 0], [0, 0, 0]]) == 2
assert answer_3_paths_mod([[0] * 300 for _ in range(300)]) == comb(598, 299) % (10**9 + 7)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [22]:
def answer_4_can_reach(grid: List[List[int]]) -> bool:
    n = len(grid[0])
    row = [False] * n
    row[0] = True
    for grid_row in grid:
        for c in range(n):
            if grid_row[c] == 1:
                row[c] = False
            elif c > 0:
                row[c] = row[c] or row[c - 1]
    return row[-1]

assert answer_4_can_reach([[0, 0, 0], [0, 1, 0], [0, 0, 0]]) is True
assert answer_4_can_reach([[0, 1], [1, 0]]) is False
assert answer_4_can_reach([[0]]) is True
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [23]:
from itertools import combinations

def answer_5_min_blocks(m: int, n: int) -> int:
    cells = [(r, c) for r in range(m) for c in range(n) if (r, c) not in ((0, 0), (m - 1, n - 1))]
    for k in range(len(cells) + 1):
        for chosen in combinations(cells, k):
            grid = [[0] * n for _ in range(m)]
            for r, c in chosen:
                grid[r][c] = 1
            if not answer_4_can_reach(grid):
                return k
    return -1

assert answer_5_min_blocks(2, 2) == 2
assert answer_5_min_blocks(3, 3) == 2
assert answer_5_min_blocks(2, 4) == 2
print("Bài 5 PASS;", {(m, n): answer_5_min_blocks(m, n) for m in range(2, 5) for n in range(2, 5)})

Bài 5 PASS; {(2, 2): 2, (2, 3): 2, (2, 4): 2, (3, 2): 2, (3, 3): 2, (3, 4): 2, (4, 2): 2, (4, 3): 2, (4, 4): 2}


Quy luật: luôn bằng **2** với `m, n >= 2` — chặn `(0, 1)` và `(1, 0)` là đủ, và một ô thì không đủ vì luôn có hai đường "vòng trên-phải" và "vòng trái-dưới" không chung ô nào (ngoài đầu và đích). Với lưới `1 × n` (n ≥ 3) thì chỉ cần 1.

# 14. Tổng kết

> Như #62, nhưng ô vật cản có **0** đường. Duyệt mọi ô bằng một công thức (cộng ô trên và ô trái nếu tồn tại), khởi tạo `row[0] = 1` — mọi trường hợp biên (vật cản ở hàng/cột đầu, ô đầu/đích bị chặn) tự được xử lý.

---

## Checklist trước khi nộp

- [ ] Ô vật cản → 0?
- [ ] Không khởi tạo cả hàng 0 / cột 0 bằng 1?
- [ ] Ô đầu / ô đích bị chặn → 0?
- [ ] Đã thử lưới `1 × 1`, một hàng có vật cản giữa?